# Data Ingestion
---

## Optical: Sentinel-2
---

In [3]:
import ee
import geopandas as gpd
import shapely.geometry
import wxee  # pip install wxee
import rioxarray

# 1. Initialize GEE with high-volume API endpoint extension via wxee
wxee.Initialize()

# 2. Define an Area of Interest (AOI) using GeoPandas
# (Replace coordinates with your BC LiDAR target region footprint)
aoi_geom = shapely.geometry.box(-123.4, 48.4, -123.3, 48.5) # Victoria, BC area
gdf = gpd.GeoDataFrame(index=[0], crs="EPSG:4326", geometry=[aoi_geom])

# Convert GeoPandas geometry to Earth Engine Geometry
ee_aoi = ee.Geometry.Polygon(list(gdf.geometry.iloc[0].exterior.coords))

# 3. Create a Sentinel-2 Cloud-Free Composite on the Cloud
def mask_s2_clouds(image):
    qa = image.select('QA60')
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11
    mask = qa.bitwiseAnd(cloud_bit_mask).eq(0).and_(
           qa.bitwiseAnd(cirrus_bit_mask).eq(0))
    return image.updateMask(mask).divide(10000)

# Filter collection for summer 2026 / matching LiDAR epoch
s2_col = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
          .filterBounds(ee_aoi)
          .filterDate('2026-06-01', '2026-08-31')
          .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
          .map(mask_s2_clouds)
          .select(['B2', 'B3', 'B4', 'B8'])) # Blue, Green, Red, NIR

# Create a median composite to collapse the time dimension for structure analysis
s2_img = s2_col.median().clip(ee_aoi)

# 4. Ingest directly into Xarray using wxee
print("Streaming Earth Engine image to local Xarray Dataset...")
ds = s2_img.wx.to_xarray(scale=10, region=ee_aoi) # 10m target resolution

# 5. Bring in rioxarray spatial accessors for downstream ML preprocessing
ds = ds.rio.write_crs("EPSG:4326")

print("\n--- Pipeline Target Array Acquired ---")
print(ds)

AttributeError: 'Image' object has no attribute 'and_'

In [4]:
import ee
import geemap
import xarray as xr
import rioxarray as rxr

In [6]:
ee.Authenticate()
ee.Initialize(
    project="multimodal-regression"
)

In [11]:
# Manually select AoI
m = geemap.Map(basemap="SATELLITE")
m

Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', transp…

## SAR: Sentinel-1 C-Band
---